In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "analysis" / "encoder_roundtrip_table1.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not locate analysis/encoder_roundtrip_table1.py from the current working directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")


PROJECT_ROOT = /data/home/umang/Materials/Reynolds-QSR


# Table 1: Encoder round-trip fidelity

This notebook uses only the **local-isometric encoder** and the **cubochoric lookup/refinement decoder**.

Flow used for this task:

1. Load one HR test scan.
2. Flatten the scan to passive quaternions with shape `(N, 4)`.
3. Encode with `LocalIsoCrystalEncoder.forward_a1(...)`.
4. Decode with `CubochoricOptimizingLocalIsoDecoder(...)`.
5. Compute symmetry-aware `dS` in radians.
6. Average `dS` within each scan, then report `mean ± std` across scans.

No SR forward pass, no upsampling path, and no full model checkpoint are used here.

In [2]:
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

from analysis.encoder_roundtrip_table1 import (
    default_table1_datasets,
    default_table1_variants,
    describe_roundtrip_flow,
    evaluate_roundtrip,
    pivot_paper_table,
    summarize_scan_metrics,
)

/data/home/umang/miniconda3/envs/material/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
datasets = default_table1_datasets()
variants = default_table1_variants()

print("Flow used for this task:")
for step_idx, step in enumerate(describe_roundtrip_flow(), start=1):
    print(f"{step_idx}. {step}")

device = "cuda" if torch.cuda.is_available() else "cpu"
take_first = None  # set to a small integer like 4 for a smoke test
chunk_size = 4096 if device.startswith("cuda") else 1024

output_dir = PROJECT_ROOT / "analysis" / "out" / "table1_roundtrip"
output_dir.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([spec.__dict__ for spec in datasets]))
display(pd.DataFrame([variant.__dict__ for variant in variants]))

print(f"device={device}")
print(f"take_first={take_first}")
print(f"chunk_size={chunk_size}")


Flow used for this task:
1. load one HR test scan
2. flatten the scan to passive quaternions with shape (N, 4)
3. encode quaternions with LocalIsoCrystalEncoder.forward_a1(...)
4. decode features with CubochoricOptimizingLocalIsoDecoder(...)
5. compute symmetry-aware dS(q_decoded, q_ref) in radians
6. average dS within each scan, then report mean +/- std across scans


,label,dataset_root,crystal,d6_convention,model_module,split,decoder_cubochoric_resolution,decoder_method,decoder_table_cache_dir
0,IN718 (FCC),/data/home/umang/Materials/Materials_data_moun...,fcc,z_axis,models.SR_double_conv_SRattn_a1,Test,1,cubochoric,out/decoder_lookup_tables/fcc
1,Ti-6Al-4V (HCP),/data/home/umang/Materials/Materials_data_moun...,hcp,z_axis,models.SR_double_conv_SRattn_a1,Test,1,cubochoric,out/decoder_lookup_tables/hcp


,label,num_starts,steps,lr
0,LUT lookup only (no refinement),1,0,0.03
1,"LUT + local refinement (12 steps, 8 starts)",8,12,0.03


device=cuda
take_first=None
chunk_size=4096


In [4]:
df= pd.DataFrame([spec.__dict__ for spec in datasets])
df['dataset_root'][0], df['dataset_root'][1]

('/data/home/umang/Materials/Materials_data_mount/datasets/IN718',
 '/data/home/umang/Materials/Materials_data_mount/datasets/Ti_Al_1pct_QSR_x4')

In [5]:
scan_frames = []
for spec in datasets:
    for variant in variants:
        scan_frames.append(
            evaluate_roundtrip(
                spec,
                variant,
                device=device,
                take_first=take_first,
                chunk_size=chunk_size,
                progress=True,
            )
        )

scan_df = pd.concat(scan_frames, ignore_index=True)
summary_df = summarize_scan_metrics(scan_df)
table_df = pivot_paper_table(summary_df)

scan_csv = output_dir / "scan_metrics.csv"
summary_csv = output_dir / "summary_metrics.csv"
table_csv = output_dir / "paper_table.csv"

scan_df.to_csv(scan_csv, index=False)
summary_df.to_csv(summary_csv, index=False)
table_df.to_csv(table_csv, index=False)

print(f"wrote {scan_csv}")
print(f"wrote {summary_csv}")
print(f"wrote {table_csv}")


IN718 (FCC) | LUT lookup only (no refinement): 100%|██████████| 147/147 [04:51<00:00,  1.98s/it]
IN718 (FCC) | LUT + local refinement (12 steps, 8 starts): 100%|██████████| 147/147 [06:01<00:00,  2.46s/it]
Ti-6Al-4V (HCP) | LUT lookup only (no refinement): 100%|██████████| 133/133 [07:03<00:00,  3.18s/it]
Ti-6Al-4V (HCP) | LUT + local refinement (12 steps, 8 starts): 100%|██████████| 133/133 [05:55<00:00,  2.67s/it]

wrote /data/home/umang/Materials/Reynolds-QSR/analysis/out/table1_roundtrip/scan_metrics.csv
wrote /data/home/umang/Materials/Reynolds-QSR/analysis/out/table1_roundtrip/summary_metrics.csv
wrote /data/home/umang/Materials/Reynolds-QSR/analysis/out/table1_roundtrip/paper_table.csv


In [6]:
display(summary_df.sort_values(["variant_label", "dataset_label"]))

paper_table = table_df.rename(columns={"variant_label": "Variant"})
display(paper_table)

paper_table


,dataset_label,variant_label,num_scans,mean_of_scan_mean_dS_rad,std_of_scan_mean_dS_rad,weighted_mean_dS_rad,paper_value
1,IN718 (FCC),"LUT + local refinement (12 steps, 8 starts)",147,0.011785,0.000980,0.011785,0.0118 +/- 0.0010
3,Ti-6Al-4V (HCP),"LUT + local refinement (12 steps, 8 starts)",133,0.018284,0.001202,0.018284,0.0183 +/- 0.0012
0,IN718 (FCC),LUT lookup only (no refinement),147,0.007620,0.000552,0.007620,0.0076 +/- 0.0006
2,Ti-6Al-4V (HCP),LUT lookup only (no refinement),133,0.007922,0.000618,0.007922,0.0079 +/- 0.0006


dataset_label,Variant,IN718 (FCC),Ti-6Al-4V (HCP)
0,"LUT + local refinement (12 steps, 8 starts)",0.0118 +/- 0.0010,0.0183 +/- 0.0012
1,LUT lookup only (no refinement),0.0076 +/- 0.0006,0.0079 +/- 0.0006


dataset_label,Variant,IN718 (FCC),Ti-6Al-4V (HCP)
0,"LUT + local refinement (12 steps, 8 starts)",0.0118 +/- 0.0010,0.0183 +/- 0.0012
1,LUT lookup only (no refinement),0.0076 +/- 0.0006,0.0079 +/- 0.0006
